In [7]:

import polars as pl

In [24]:
INPUT_FILE = "ad_data.csv"
TOP_K = 10

# Explicit schema for faster parsing and memory efficiency
SCHEMA = {
    "campaign_id": pl.Utf8,
    "date": pl.Utf8,
    "impressions": pl.Int64,
    "clicks": pl.Int64,
    "spend": pl.Float64,
    "conversions": pl.Int64,
}

In [25]:
lf = pl.scan_csv(
    INPUT_FILE,
    schema=SCHEMA,
    low_memory=True
)

lf

In [26]:
agg = (
    lf.group_by("campaign_id")
    .agg(
        [
            pl.sum("impressions").alias("total_impressions"),
            pl.sum("clicks").alias("total_clicks"),
            pl.sum("spend").alias("total_spend"),
            pl.sum("conversions").alias("total_conversions"),
        ]
    )
    .with_columns(
        [
            pl.when(pl.col("total_impressions") > 0)
            .then(pl.col("total_clicks") / pl.col("total_impressions"))
            .otherwise(None)
            .alias("CTR"),

            pl.when(pl.col("total_conversions") > 0)
            .then(pl.col("total_spend") / pl.col("total_conversions"))
            .otherwise(None)
            .alias("CPA"),
        ]
    )
)

agg

In [27]:
top_ctr = (
    agg
    .top_k(TOP_K, by="CTR")
    .with_columns(
        [
            pl.col("CTR").round(4),
            pl.col("CPA").round(2)
        ]
    )
    .collect(streaming=True)
)

top_ctr

/tmp/ipykernel_3020/1311664263.py:2: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  agg


campaign_id,total_impressions,total_clicks,total_spend,total_conversions,CTR,CPA
str,i64,i64,f64,i64,f64,f64
"""CMP005""",13648608306,375627610,3.9478e8,20403485,0.0275,19.35
"""CMP022""",13703365735,377109904,3.9590e8,20480486,0.0275,19.33
"""CMP023""",13657508417,375769033,3.9420e8,20390213,0.0275,19.33
"""CMP011""",13714973625,377330481,3.9631e8,20463005,0.0275,19.37
"""CMP018""",13708338704,377076026,3.9600e8,20477639,0.0275,19.34
"""CMP026""",13652760288,375491308,3.9444e8,20370126,0.0275,19.36
"""CMP019""",13675562649,376096719,3.9506e8,20415450,0.0275,19.35
"""CMP042""",13717935895,377260350,3.9658e8,20507021,0.0275,19.34
"""CMP004""",13705584584,376906722,3.9599e8,20491770,0.0275,19.32


In [19]:
top_cpa = (
    agg
    .filter(pl.col("total_conversions") > 0)
    .top_k(TOP_K, by="CPA", descending=False)
    .with_columns(
        [
            pl.col("CTR").round(4),
            pl.col("CPA").round(2)
        ]
    )
    .collect(streaming=True)
)

top_cpa

/tmp/ipykernel_3020/4174837967.py:2: DeprecationWarning: the argument `descending` for `LazyFrame.top_k` is deprecated. It was renamed to `reverse` in version 1.0.0.
  agg
/tmp/ipykernel_3020/4174837967.py:2: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  agg


campaign_id,total_impressions,total_clicks,total_spend,total_conversions,CTR,CPA
str,i64,i64,f64,i64,f64,f64
"""CMP035""",13689287094,376229378,3.9562e8,20399031,0.0275,19.39
"""CMP012""",13679171528,375898116,3.9541e8,20392719,0.0275,19.39
"""CMP010""",13699272787,376409406,3.9574e8,20412776,0.0275,19.39
"""CMP021""",13687991449,375796619,3.9504e8,20382306,0.0275,19.38
"""CMP011""",13714973625,377330481,3.9631e8,20463005,0.0275,19.37
"""CMP025""",13688208851,375831578,3.9516e8,20406444,0.0275,19.36
"""CMP038""",13711890563,376608656,3.9584e8,20441954,0.0275,19.36
"""CMP026""",13652760288,375491308,3.9444e8,20370126,0.0275,19.36
"""CMP014""",13688374487,376224010,3.9554e8,20428052,0.0275,19.36


In [ ]:
import polars as pl

INPUT_FILE = "ad_data.csv"

lf = (
    pl.scan_csv(
        INPUT_FILE,
        infer_schema_length=1000,
        low_memory=True
    )
)
agg = (
    lf.group_by("campaign_id")
    .agg(
        [
            pl.col("impressions").sum().alias("total_impressions"),
            pl.col("clicks").sum().alias("total_clicks"),
            pl.col("spend").sum().alias("total_spend"),
            pl.col("conversions").sum().alias("total_conversions"),
        ]
    )
    .with_columns(
        [
            (pl.col("total_clicks") / pl.col("total_impressions")).alias("CTR"),
            pl.when(pl.col("total_conversions") > 0)
            .then(pl.col("total_spend") / pl.col("total_conversions"))
            .otherwise(None)
            .alias("CPA"),
        ]
    )
)

In [ ]:
top_ctr = (
    agg
    .top_k(10, by="CTR")
    .collect(streaming=True)
)

top_ctr.write_csv("top10_ctr.csv")

top_cpa = (
    agg
    .filter(pl.col("total_conversions") > 0)
    .top_k(10, by="CPA", reverse=True)
    .collect(streaming=True)
)
top_cpa.write_csv("top10_cpa.csv")

/tmp/ipykernel_3020/1620197220.py:2: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  agg.sort("CTR", descending=True)
/tmp/ipykernel_3020/1620197220.py:10: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  agg.filter(pl.col("total_conversions") > 0)
